# Day 062 — Exercise 4: Regression Test

A **regression test** documents a bug so it can never silently come back. The workflow:

1. Discover a bug
2. Write a test that FAILS (proving the bug exists)
3. Fix the bug
4. Confirm the test now PASSES
5. The test lives in the suite forever — it will catch any future regression

The bug here: `GET /multiply?a=3&b=4` returns `7` (the sum) instead of `12` (the product). Your test must catch this specific mistake.

In [ ]:
from fastapi import FastAPI
from starlette.testclient import TestClient

def _build_broken_api():
    """Buggy API: /multiply returns the SUM, not the product."""
    app = FastAPI()
    @app.get("/multiply")
    def multiply(a: int, b: int):
        return {"result": a + b}    # BUG: should be a * b
    return app

def _build_fixed_api():
    """Fixed API: /multiply returns the product."""
    app = FastAPI()
    @app.get("/multiply")
    def multiply(a: int, b: int):
        return {"result": a * b}
    return app


In [ ]:
# no extra imports needed


## Task

Implement `test_multiply_correct(client) -> None`:

- `GET /multiply` with query params `a=3, b=4`
- Assert `status_code == 200`
- Assert `result == 12` — include the actual value in the failure message

The test should **FAIL** against `_build_broken_api()` (returns 7) and **PASS** against `_build_fixed_api()` (returns 12).

## Your Implementation

In [ ]:
def test_multiply_correct(client) -> None:
    """Test that GET /multiply?a=3&b=4 returns {result: 12}.

    Use client.get('/multiply', params={'a': 3, 'b': 4}).
    Assert status is 200.
    Assert result == 12 (include actual value in assertion message).
    This test should FAIL against the broken API and PASS against the fixed one.
    """
    # TODO: make the request, assert 200, assert result == 12
    raise NotImplementedError


In [ ]:
def test_multiply_correct(client) -> None:
    r = client.get("/multiply", params={"a": 3, "b": 4})
    assert r.status_code == 200
    data = r.json()
    assert data["result"] == 12, (
        f"Expected 12 (3*4), got {data['result']} (may be 3+4=7 if the bug is present)")


## Automated checks

In [ ]:
score, total = 0, 4
try:
    # test raises AssertionError against the broken API (catches the bug)
    broken = TestClient(_build_broken_api(), raise_server_exceptions=False)
    try:
        test_multiply_correct(broken)
        print("\u274c test_multiply_correct did NOT catch the bug")
    except AssertionError:
        score += 1; print("\u2705 test_multiply_correct FAILS against the buggy API")
    except NotImplementedError:
        print("\u274c function raises NotImplementedError — implement it first")

    # test passes against the fixed API
    fixed = TestClient(_build_fixed_api(), raise_server_exceptions=False)
    try:
        test_multiply_correct(fixed)
        score += 1; print("\u2705 test_multiply_correct PASSES against the fixed API")
    except Exception as e:
        print(f"\u274c test_multiply_correct raised on fixed API: {e}")

    # test asserts status 200
    try:
        class _FakeResp:
            status_code = 503
            def json(self): return {"result": 12}
        class _FakeClient:
            def get(self, *a, **kw): return _FakeResp()
        test_multiply_correct(_FakeClient())
        print("\u274c should have raised on 503")
    except AssertionError:
        score += 1; print("\u2705 test checks status_code (caught 503)")
    except NotImplementedError:
        print("\u274c not implemented")
    except Exception:
        score += 1; print("\u2705 test rejects 503 (some exception raised)")

    # test asserts result == 12 specifically
    broken_12 = TestClient(_build_broken_api(), raise_server_exceptions=False)
    # a=6, b=6 → broken returns 12 (6+6), fixed returns 36 (6*6)
    # So if we test a=3, b=4 → broken returns 7 which != 12 ✓
    try:
        test_multiply_correct(broken)  # a=3,b=4 → broken returns 7
        print("\u274c bug not caught for 3*4=12")
    except AssertionError as e:
        assert "12" in str(e) or "7" in str(e) or "result" in str(e).lower(), (
            f"Assertion message should reference 12 or 7: {e!r}")
        score += 1; print("\u2705 assertion message references expected/actual values")
    except NotImplementedError:
        print("\u274c not implemented")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def test_multiply_correct(client) -> None:
    r = client.get("/multiply", params={"a": 3, "b": 4})
    assert r.status_code == 200
    data = r.json()
    assert data["result"] == 12, (
        f"Expected 12 (3*4), got {data['result']} (may be 3+4=7 if the bug is present)")
```

**Include actual values in assertion messages.** `assert result == 12` prints only `AssertionError`. `assert result == 12, f'Expected 12, got {result}'` immediately tells you the actual value — crucial when tests run in CI at 2am and you're reading a log, not a debugger.

</details>